# Project 2

##### Imports

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
import kagglehub

c:\Users\fbase\All_vscode_projects\envs\ml-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


##### 1. Run this section to load dataset

$$\huge \texttt{Data loading}$$

In [3]:
# Run this once for downloading the dataset
#url = "https://data.cityofnewyork.us/resource/erm2-nwe9.csv?$limit=200000"
#df = pd.read_csv(url)

#df.to_csv("nyc_311_sample.csv", index=False)

In [4]:
df = pd.read_csv("nyc_311_sample.csv")
df.head()

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,incident_zip,...,vehicle_type,taxi_company_borough,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,road_ramp,bridge_highway_segment,latitude,longitude,location
0,68509446,2026-03-31T02:39:38.000,NaN,DOT,Department of Transportation,Street Condition,Pothole,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,68506247,2026-03-31T02:30:56.000,NaN,DOT,Department of Transportation,Street Condition,Pothole,NaN,NaN,10031.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,68503135,2026-03-31T02:26:57.000,NaN,DOT,Department of Transportation,Street Condition,Pothole,NaN,NaN,11223.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.601042,-73.979387,POINT (-73.979387264647 40.601042365906)
3,68501976,2026-03-31T02:26:00.000,2026-03-31T02:59:00.000,DSNY,Department of Sanitation,Derelict Vehicles,Derelict Vehicles,NaN,Street,11373.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.743346,-73.886114,POINT (-73.886113509027 40.74334566695)
4,68509447,2026-03-31T02:17:11.000,NaN,DOT,Department of Transportation,Street Condition,Pothole,NaN,NaN,11101.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
clean_status = pd.DataFrame([],columns=df.columns)
clean_status[df.columns] = np.full((1,44),False)

In [6]:
clean_status

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,incident_zip,...,vehicle_type,taxi_company_borough,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,road_ramp,bridge_highway_segment,latitude,longitude,location
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


##### 2. Take a peek at your dataset

$$\huge \texttt{Data overview}$$


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 44 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   unique_key                      200000 non-null  int64  
 1   created_date                    200000 non-null  str    
 2   closed_date                     158957 non-null  str    
 3   agency                          200000 non-null  str    
 4   agency_name                     200000 non-null  str    
 5   complaint_type                  200000 non-null  str    
 6   descriptor                      197180 non-null  str    
 7   descriptor_2                    74933 non-null   str    
 8   location_type                   162834 non-null  str    
 9   incident_zip                    197358 non-null  float64
 10  incident_address                187723 non-null  str    
 11  street_name                     187719 non-null  str    
 12  cross_street_1             

In [8]:
df.isna().sum()

unique_key                             0
created_date                           0
closed_date                        41043
agency                                 0
agency_name                            0
complaint_type                         0
descriptor                          2820
descriptor_2                      125067
location_type                      37166
incident_zip                        2642
incident_address                   12277
street_name                        12281
cross_street_1                     63730
cross_street_2                     63705
intersection_street_1              72611
intersection_street_2              72529
address_type                        1705
city                               10626
landmark                           91597
facility_type                     199346
status                                 0
due_date                          199254
resolution_description             13782
resolution_action_updated_date     12180
community_board 

In [9]:
"""
for i in range(1,5):
    try:
        display(df.iloc[:,i:i*10].head())
    except:
        display(df.iloc[:,i*10:].head())
"""
#Looking at few columns makes it easier
subset1 = df.iloc[:,:11].head()
subset1

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,incident_zip,incident_address
0,68509446,2026-03-31T02:39:38.000,NaN,DOT,Department of Transportation,Street Condition,Pothole,NaN,NaN,NaN,65 ROAD
1,68506247,2026-03-31T02:30:56.000,NaN,DOT,Department of Transportation,Street Condition,Pothole,NaN,NaN,10031.0,CONVENT AVENUE
2,68503135,2026-03-31T02:26:57.000,NaN,DOT,Department of Transportation,Street Condition,Pothole,NaN,NaN,11223.0,NaN
3,68501976,2026-03-31T02:26:00.000,2026-03-31T02:59:00.000,DSNY,Department of Sanitation,Derelict Vehicles,Derelict Vehicles,NaN,Street,11373.0,42-71 79 STREET
4,68509447,2026-03-31T02:17:11.000,NaN,DOT,Department of Transportation,Street Condition,Pothole,NaN,NaN,11101.0,43 STREET


In [10]:
subset2 = df.iloc[:,11:21].head()
subset2

,street_name,cross_street_1,cross_street_2,intersection_street_1,intersection_street_2,address_type,city,landmark,facility_type,status
0,65 ROAD,POWER ROAD,102 STREET,NaN,NaN,BLOCKFACE,NaN,NaN,NaN,Open
1,CONVENT AVENUE,WEST 133 STREET,WEST 135 STREET,NaN,NaN,BLOCKFACE,MANHATTAN,NaN,NaN,Open
2,NaN,NaN,NaN,AVENUE S,WEST 7 STREET,INTERSECTION,BROOKLYN,NaN,NaN,Open
3,79 STREET,WOODSIDE AVENUE,45 AVENUE,NaN,NaN,ADDRESS,ELMHURST,NaN,NaN,Closed
4,43 STREET,35 AVENUE,NORTHERN BOULEVARD,NaN,NaN,BLOCKFACE,QUEENS,NaN,NaN,Open


In [11]:
subset3 = df.iloc[:,21:31].head()
subset3

,due_date,resolution_description,resolution_action_updated_date,community_board,council_district,police_precinct,bbl,borough,x_coordinate_state_plane,y_coordinate_state_plane
0,NaN,The Department of Transportation referred this...,2026-03-31T02:39:38.000,Unspecified QUEENS,NaN,Unspecified,NaN,QUEENS,NaN,NaN
1,NaN,The Department of Transportation referred this...,2026-03-31T02:30:56.000,09 MANHATTAN,NaN,Precinct 26,NaN,MANHATTAN,NaN,NaN
2,NaN,The Department of Transportation referred this...,2026-03-31T02:26:57.000,11 BROOKLYN,43.0,Precinct 62,NaN,BROOKLYN,989974.0,158253.0
3,NaN,DSNY already has a request to investigate this...,2026-03-31T12:00:00.000,04 QUEENS,25.0,Precinct 110,4.015250e+09,QUEENS,1015808.0,210118.0
4,NaN,The Department of Transportation referred this...,2026-03-31T02:17:11.000,01 QUEENS,NaN,Precinct 114,NaN,QUEENS,NaN,NaN


In [12]:
subset4 = df.iloc[:,31:].head()
subset4

,open_data_channel_type,park_facility_name,park_borough,vehicle_type,taxi_company_borough,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,road_ramp,bridge_highway_segment,latitude,longitude,location
0,UNKNOWN,Unspecified,QUEENS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,UNKNOWN,Unspecified,MANHATTAN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,UNKNOWN,Unspecified,BROOKLYN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.601042,-73.979387,POINT (-73.979387264647 40.601042365906)
3,PHONE,Unspecified,QUEENS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.743346,-73.886114,POINT (-73.886113509027 40.74334566695)
4,UNKNOWN,Unspecified,QUEENS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


##### 3. Cleaning

$$\huge \texttt{Data Cleaning}$$

###### Things I could do next time
1. Evaluate where we can remove NaN from
2. Seperate time from datetime columns
3. Evaluate the values of all columns do they all make sense or is there corrupted data

In [13]:
subset1.head()

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,incident_zip,incident_address
0,68509446,2026-03-31T02:39:38.000,NaN,DOT,Department of Transportation,Street Condition,Pothole,NaN,NaN,NaN,65 ROAD
1,68506247,2026-03-31T02:30:56.000,NaN,DOT,Department of Transportation,Street Condition,Pothole,NaN,NaN,10031.0,CONVENT AVENUE
2,68503135,2026-03-31T02:26:57.000,NaN,DOT,Department of Transportation,Street Condition,Pothole,NaN,NaN,11223.0,NaN
3,68501976,2026-03-31T02:26:00.000,2026-03-31T02:59:00.000,DSNY,Department of Sanitation,Derelict Vehicles,Derelict Vehicles,NaN,Street,11373.0,42-71 79 STREET
4,68509447,2026-03-31T02:17:11.000,NaN,DOT,Department of Transportation,Street Condition,Pothole,NaN,NaN,11101.0,43 STREET


Columns : Reason why null values appear
Closed dates : Because those service records have not been resolved?
>Does a service request being closed always mean that it has been resolved ( yes , no cause of something else , no without any reason ) 

In [14]:
"""
Q1 = [['question','answer']]
print("Try to answer in one line if possible")
for i in df.columns:
    question = f"How could {i} be the reason for closing a Service request?"
    print(question)
    answer = input(" Answer :")
    Q1.append([question,answer])

td = pd.DataFrame(Q1)
td.to_csv("Inventory.csv",columns=['question','answer'])
"""

'\nQ1 = [[\'question\',\'answer\']]\nprint("Try to answer in one line if possible")\nfor i in df.columns:\n    question = f"How could {i} be the reason for closing a Service request?"\n    print(question)\n    answer = input(" Answer :")\n    Q1.append([question,answer])\n\ntd = pd.DataFrame(Q1)\ntd.to_csv("Inventory.csv",columns=[\'question\',\'answer\'])\n'

1. A specific location/region could be the reason for closing or request
2. A specific type of problem could be the description for closing a request
3. A specific entity could be the reason for closing a request ( The work well so close more SR )

In [15]:
df['status'].unique()

<StringArray>
[       'Open',      'Closed', 'In Progress',    'Assigned',     'Started',
     'Pending', 'Unspecified']
Length: 7, dtype: str

In [16]:
cond1 = [df['closed_date'].notna(),df['closed_date'].isna()]
cd = ['not_null',np.nan]
ts = []
for i in range(2):
    for j in df['status'].unique():
        td = df[cond1[i]]
        try:
            td[td['status']==j].iloc[0]
            ts.append([cd[i],j])
        except:
            pass
            ts.append([cd[i],np.nan])

closed_date_x_status = pd.DataFrame(ts,columns=['closed_date','status'])
closed_date_x_status
            

,closed_date,status
0,not_null,Open
1,not_null,Closed
2,not_null,NaN
3,not_null,Assigned
4,not_null,Started
5,not_null,Pending
6,not_null,Unspecified
7,NaN,Open
8,NaN,NaN
9,NaN,In Progress


In [17]:
notnull_closed_dates = df[df['closed_date'].notna()]['closed_date']
pattern = r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d+\.\d+$"
notnull_closed_dates.str.match(pattern).unique()

array([ True])

In [18]:
notnull_closed_dates.shape[0] + df[df['closed_date'].isna()].shape[0]
# This proves that except for the NaN ,  closed date has all valid string which follow this pattern (no rubbish value like unknown etc )

200000

In [19]:
notnull_closed_dates[notnull_closed_dates.duplicated(keep=False)]


75        2026-03-31T02:01:26.000
109       2026-03-31T01:54:57.000
159       2026-03-31T01:00:21.000
201       2026-03-31T01:54:57.000
203       2026-03-31T01:29:50.000
                   ...           
199965    2026-03-30T19:37:43.000
199973    2026-03-14T15:54:00.000
199983    2026-03-13T00:00:00.000
199985    2026-03-12T12:59:01.000
199999    2026-03-13T08:56:36.000
Name: closed_date, Length: 51306, dtype: str

In [20]:
value = '2026-03-31T02:01:26.000'
notnull_closed_dates[notnull_closed_dates.str.match(value)]
# Okay there are some duplicated values , is this possible?

75      2026-03-31T02:01:26.000
1796    2026-03-31T02:01:26.000
Name: closed_date, dtype: str

In [21]:
with pd.option_context('display.max_rows', None,
                       'display.max_columns', None):
    display(df.iloc[[75,1796],:])
# Okay two entries can be closed at the same time , sounds rare but its possible 

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,incident_zip,incident_address,street_name,cross_street_1,cross_street_2,intersection_street_1,intersection_street_2,address_type,city,landmark,facility_type,status,due_date,resolution_description,resolution_action_updated_date,community_board,council_district,police_precinct,bbl,borough,x_coordinate_state_plane,y_coordinate_state_plane,open_data_channel_type,park_facility_name,park_borough,vehicle_type,taxi_company_borough,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,road_ramp,bridge_highway_segment,latitude,longitude,location
75,68504007,2026-03-31T01:23:55.000,2026-03-31T02:01:26.000,NYPD,New York City Police Department,Noise - Commercial,Loud Talking,NaN,Club/Bar/Restaurant,11216.0,299 HALSEY STREET,HALSEY STREET,TOMPKINS AVENUE,THROOP AVENUE,TOMPKINS AVENUE,THROOP AVENUE,ADDRESS,BROOKLYN,HALSEY STREET,NaN,Closed,NaN,The New York City Police Department responded ...,2026-03-31T02:01:12.000,03 BROOKLYN,36.0,Precinct 79,3.018400e+09,BROOKLYN,1000520.0,188042.0,ONLINE,Unspecified,BROOKLYN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.682794,-73.941338,POINT (-73.94133816146 40.682793665961)
1796,68511460,2026-03-30T21:09:03.000,2026-03-31T02:01:26.000,NYPD,New York City Police Department,Illegal Parking,Blocked Crosswalk,NaN,Street/Sidewalk,11219.0,42 STREET,42 STREET,42 STREET,10 AVENUE,42 STREET,10 AVENUE,INTERSECTION,NaN,NaN,NaN,Closed,NaN,The New York City Police Department responded ...,2026-03-31T02:01:21.000,12 BROOKLYN,38.0,Precinct 66,NaN,BROOKLYN,985737.0,173516.0,ONLINE,Unspecified,BROOKLYN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.642938,-73.994642,POINT (-73.994641792031 40.642937837713)


In [22]:
print(df.duplicated().unique())
print(df.unique_key.duplicated().unique())
# No duplicate rows / entries

[False]
[False]


`closed_date` is completely clean

In [23]:
# Now are all these combinations of closed_data and status possible?
closed_date_x_status['Possible?'] = [0,1,1,0,0,0,1,1,0,1,1,1,1,0]
closed_date_x_status['Possible?'] = closed_date_x_status['Possible?'].astype('bool')
closed_date_x_status['description'] = ["Human error","Possible","Human error","Human error","Human error",
                                       "Human error","Human error","Possible","Human error","Possible",
                                       "Possible","Possible","Possible","Human error"]
closed_date_x_status
# This will be useful when we are using status column as our feature

,closed_date,status,Possible?,description
0,not_null,Open,False,Human error
1,not_null,Closed,True,Possible
2,not_null,NaN,True,Human error
3,not_null,Assigned,False,Human error
4,not_null,Started,False,Human error
5,not_null,Pending,False,Human error
6,not_null,Unspecified,True,Human error
7,NaN,Open,True,Possible
8,NaN,NaN,False,Human error
9,NaN,In Progress,True,Possible


In [24]:
#  1 means clean
#  2 means not clean
#  3 means that its not clean but I know which ones are not clean
# -1 means we assume its clean , because we are not gonna use it 
#  Boolean values mean , I haven't gone through them
clean_status[['unique_key','closed_date','status']] = [1,1,3]

In [25]:
print(df[df['created_date'].isna()].size)
# There are no null values in created_date

pattern = r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d+\.\d+$"


print(df['created_date'].str.match(pattern).unique()) # gives a boolean array where the pattern matches, False implies there are values which don't match this pattern , True implies there are values which match this pattern

print("No of duplicated values :",df['created_date'][df['created_date'].duplicated(keep=False)].size)
print("Total values : ",df['created_date'].shape[0])


0
[ True]
No of duplicated values : 55830
Total values :  200000


In [26]:
clean_status['created_date'] = 1

In [27]:
b = {k:df.iloc[:,k].unique() for k in [3,4,5]}
for k,v in b.items():
    print(k)
    print(v,'\n')


3
<StringArray>
[  'DOT',  'DSNY',  'NYPD',   'DHS',  'DCWP',   'OOS',   'OTI', 'DOHMH',
   'TLC',   'DPR',   'HPD',   'DEP',   'EDC',   'DOB',   'DOE']
Length: 15, dtype: str 

4
<StringArray>
[                      'Department of Transportation',
                           'Department of Sanitation',
                    'New York City Police Department',
                    'Department of Homeless Services',
       'Department of Consumer and Worker Protection',
                              'Office of the Sheriff',
                'Office of Technology and Innovation',
            'Department of Health and Mental Hygiene',
                      'Taxi and Limousine Commission',
                 'Department of Parks and Recreation',
 'Department of Housing Preservation and Development',
             'Department of Environmental Protection',
                   'Economic Development Corporation',
                            'Department of Buildings',
                            'Departm

In [28]:
for i in b[5]:
    print(i)

Street Condition
Derelict Vehicles
Illegal Parking
Noise - Street/Sidewalk
Traffic
Noise - Residential
Drug Activity
Noise - Vehicle
Blocked Driveway
Dirty Condition
Homeless Person Assistance
Consumer Complaint
Non-Emergency Police Matter
Noise - Park
Noise - Commercial
Cannabis Retailer
Urinating in Public
LinkNYC
Smoking or Vaping
Lost Property
Indoor Air Quality
Graffiti
Litter Basket Request
Curb Condition
Missed Collection
Encampment
Illegal Dumping
Abandoned Vehicle
Animal-Abuse
Drinking
Rodent
Street Light Condition
Street Sign - Damaged
Food Establishment
Vendor Enforcement
Illegal Tree Damage
Abandoned Bike
Damaged Tree
Panhandling
Illegal Posting
Obstruction
GENERAL
For Hire Vehicle Complaint
Noise
UNSANITARY CONDITION
PLUMBING
SAFETY
HEAT/HOT WATER
Outdoor Dining
Traffic Signal Condition
Dead Animal
Sewer
ELECTRIC
Water System
Noise - Helicopter
Food Poisoning
WATER LEAK
Lead
Illegal Fireworks
FLOORING/STAIRS
Hazardous Materials
Overgrown Tree/Branches
Sidewalk Condition
DO

In [29]:
# agency , agency_name , complaint_type all have no null values
# After looking at their unique values , I can safely say that there are no corrupted values
clean_status[['agency','agency_name','complaint_type']] = [1,1,1]

In [30]:
for i in df['descriptor'].unique():
    print(i)
# All values look normal , it would be better to give this task to a LLM to validate if each value is valid
# furthermore descriptor and descriptor_2 don't have need to have value , therefore nan doesn't imply suspicious value

Pothole
Derelict Vehicles
Blocked Hydrant
Blocked Crosswalk
Loud Talking
Drag Racing
Loud Music/Party
Use Outside
Engine Idling
Banging/Pounding
No Access
Trash
Non-Chronic
Tobacco Sales
Trespassing
Suspicious Activity
Loud Television
Posted Parking Sign Violation
nan
Damaged/Defective
Allowed in Smoke Free Area
Car/Truck Music
Bag/Wallet
Ventilation
Graffiti
Car/Truck Horn
Replacement Basket
Double Parked Blocking Traffic
Human Feces
Blocked Bike Lane
Cave-in
Parking Permit Improper Use
Commercial Overnight Parking
Defacement
Use Indoor
Partial Access
Bulky Trash
Blocked Sidewalk
Removal Request
With License Plate
Other (complaint details)
Double Parked Blocking Vehicle
In Public
Rat Sighting
Fixture/Luminaire Out Of Position
Bus Stop
Street Light Out
Rodents/Insects/Garbage
Food Vendor
Roots Damaged
Chained to Public Property
Branch Cracked and Will Fall
Unauthorized Bus Layover
Broken Curb
Poster or Sign
Non-Food Vendor
Hotel or Motel
Neglected
Sidewalk Display
SIGNAGE MISSING
After

In [31]:
clean_status[['descriptor','descriptor_2']]=[-1,-1]
# -1 means we assume its clean , because we are not gonna use it 
# we have to give it some value because bool value implies we haven't touched that column

In [32]:
for i in df['location_type'].unique():
    print(i)

# Clean

nan
Street
Street/Sidewalk
Residential Building/House
Sidewalk
Park/Playground
Club/Bar/Restaurant
Store/Commercial
Residential Building
Taxi
3+ Family Apartment Building
Comercial
3+ Family Apt. Building
Curb
Other
Yard
Subway Station
Restaurant/Bar/Deli/Bakery
Business
RESIDENTIAL BUILDING
Subway
Above Address
Gutter
1-2 Family Dwelling
Park
Speed Reducer
Highway
Street/Curbside
Other (Explain Below)
Residential
Crosswalk
Bus Stop Shelter
Retail Store
3+ Family Mixed Use Building
Ferry
Mixed Use
Mobile Food Vendor
Private House
Apartment
Vacant Lot
House of Worship
Lot
Public/Unfenced Area
Traffic Island or Median
Commercial Building
Private School
Hallway
Food Establishment or Vendor
Parking Lot or Garage
Catering Service
Bridge
Alley
Vacant Building
1-3 Family Dwelling
Lobby
1-2 Family Mixed Use Building
Office Building
Non-Profit
Vehicle Lane
Residence
Day Care or Nursery
Cafeteria - Public School
Intersection
Terminal
Beach
Public Garden
School
Inside
Permanent Food Stand
Store
K

In [33]:
for i in df['incident_zip'].unique():
    print(i)

nan
10031.0
11223.0
11373.0
11101.0
10002.0
11207.0
10453.0
11435.0
11224.0
10467.0
10455.0
11412.0
11222.0
11102.0
11234.0
11368.0
10040.0
10032.0
10014.0
11209.0
10468.0
10473.0
10039.0
11233.0
10012.0
11433.0
11206.0
11106.0
11385.0
10304.0
10019.0
10460.0
11355.0
10030.0
10466.0
10001.0
10451.0
11365.0
11220.0
11217.0
10463.0
11237.0
11432.0
10459.0
11232.0
10037.0
10027.0
11216.0
11369.0
10462.0
11354.0
11379.0
10452.0
10472.0
11360.0
10454.0
10457.0
11417.0
10301.0
10458.0
11225.0
11418.0
10029.0
10011.0
10456.0
11235.0
11414.0
11370.0
11219.0
10023.0
11218.0
11226.0
11356.0
10026.0
11427.0
11361.0
11229.0
10470.0
10306.0
11230.0
11221.0
10003.0
11691.0
11201.0
11377.0
10036.0
11239.0
11413.0
10009.0
10004.0
11204.0
10128.0
11211.0
10035.0
10013.0
11210.0
11231.0
11375.0
11213.0
11203.0
11214.0
11208.0
11378.0
11434.0
10022.0
10461.0
10000.0
11420.0
11372.0
11205.0
11429.0
11040.0
10018.0
11416.0
11428.0
11364.0
10075.0
11105.0
11104.0
10007.0
10034.0
11374.0
10033.0
11358.0
1141

In [34]:
a = (df[df['incident_zip'].notnull()]['incident_zip']).unique()
if (a > 501).all() and (a < 11697).all(): # Nyc zip codes are between 00501 – 11697
    print(True)
else:
    print(a[a>14952]) 

# This zip code belongs to  Greenville, South Carolina which is not in New
# No clue what its doing here in nyc SR dataset


[29602.]


In [35]:
with pd.option_context('display.max_rows',None,
                       'display.max_columns',None):
    display(df[df['incident_zip']>14952])

# Even the city of incident is Greenville , logically this shouldn't be in nyc SR dataset but its a valid entry

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,incident_zip,incident_address,street_name,cross_street_1,cross_street_2,intersection_street_1,intersection_street_2,address_type,city,landmark,facility_type,status,due_date,resolution_description,resolution_action_updated_date,community_board,council_district,police_precinct,bbl,borough,x_coordinate_state_plane,y_coordinate_state_plane,open_data_channel_type,park_facility_name,park_borough,vehicle_type,taxi_company_borough,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,road_ramp,bridge_highway_segment,latitude,longitude,location
46272,68469749,2026-03-26T16:03:27.000,NaN,DCWP,Department of Consumer and Worker Protection,Consumer Complaint,Debt Collection Agency,Contract or Billing Dispute,Business,29602.0,NA PO BOX 1269,PO BOX 1269,NaN,NaN,NaN,NaN,NaN,GREENVILLE,NaN,NaN,In Progress,NaN,The Department of Consumer and Worker Protecti...,2026-03-27T09:56:52.000,0 Unspecified,NaN,Unspecified,NaN,Unspecified,NaN,NaN,ONLINE,Unspecified,Unspecified,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
for i in df['incident_address'].unique():
    print(i)

# All addresses look valid , I don't know if these are real addresses , we are gonna assume that they are

65 ROAD
CONVENT AVENUE
nan
42-71 79 STREET
43 STREET
ELDRIDGE STREET
74 ARLINGTON AVENUE
WEST BURNSIDE AVENUE
HARRISON AVENUE
139-22 87 DRIVE
90-36 149 STREET
3246 BAYVIEW AVENUE
JEROME AVENUE
1768 WALTON AVENUE
414 EAST 152 STREET
RYE PLACE
RISSE STREET
26 WEST STREET
454 AVENUE U
18-27 25 ROAD
1272 EAST 70 STREET
101-20 MARTENSE AVENUE
89 THAYER STREET
531 WEST 159 STREET
273 BENNETT AVENUE
2737 WEST 33 STREET
543 80 STREET
108 FIELD PLACE
2275 RANDALL AVENUE
2911 8 AVENUE
FULTON STREET
219 WEST 145 STREET
354 WADSWORTH AVENUE
2518 WEST 1 STREET
20 PRINCE STREET
108-26 159 STREET
251 BUSHWICK AVENUE
25-15 BROADWAY
801 SENECA AVENUE
1590 UNDERCLIFF AVENUE
180 PARK HILL AVENUE
81 MADISON STREET
705 9 AVENUE
121 MT HOPE PLACE
EAST TREMONT AVENUE
138-15 FRANKLIN AVENUE
121 MOUNT HOPE PLACE
WEST  137 STREET
4127 WICKHAM AVENUE
311 11 AVENUE
209 JAMAICA AVENUE
929 COURTLANDT AVENUE
188-35 69 AVENUE
68 STREET
310 LIVINGSTON STREET
2676 HEATH AVENUE
188-25 71 CRESCENT
GATES AVENUE
86 ROAD
89

Can location_type , incident zip , incident_address be empty?

1. location type is described as "Describes the type of location used in the address information" , Its dependent on incident address so it can be empty
2. incident zip is also dependent on incident address therefore it can be empty in cases such as the address provided isn't complete ,miss spelled or doesn't exist
3. incident_address can be empty if other incident addresses are provided. like landmark , facility_type , taxi , bridge . 
I'll look into these at last lets look at independent columns first , those are easier to clean / judge

In [37]:
na_incident_zip = df['incident_zip'].isna()
with pd.option_context('display.max_rows',None,
                       'display.max_columns',None):
    display(df.loc[na_incident_zip].head())

canTellZip = ['location_type','incident_zip','incident_address','street_name',
              'cross_street1','cross_street2','intersection_street_1','intersection_street_2',
              'address_type','city','landmark','facility_type','community_board','council_district',
              'police_precinct','bbl','borough','x_coordinate_state_plane','y_coordinate_state_plane',
              'park_facility_name','park_borough','vehicle_type','taxi_company_borough',
              'taxi_pick_up_location']

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,incident_zip,incident_address,street_name,cross_street_1,cross_street_2,intersection_street_1,intersection_street_2,address_type,city,landmark,facility_type,status,due_date,resolution_description,resolution_action_updated_date,community_board,council_district,police_precinct,bbl,borough,x_coordinate_state_plane,y_coordinate_state_plane,open_data_channel_type,park_facility_name,park_borough,vehicle_type,taxi_company_borough,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,road_ramp,bridge_highway_segment,latitude,longitude,location
0,68509446,2026-03-31T02:39:38.000,NaN,DOT,Department of Transportation,Street Condition,Pothole,NaN,NaN,NaN,65 ROAD,65 ROAD,POWER ROAD,102 STREET,NaN,NaN,BLOCKFACE,NaN,NaN,NaN,Open,NaN,The Department of Transportation referred this...,2026-03-31T02:39:38.000,Unspecified QUEENS,NaN,Unspecified,NaN,QUEENS,NaN,NaN,UNKNOWN,Unspecified,QUEENS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,68504665,2026-03-31T01:58:12.000,NaN,DOT,Department of Transportation,Street Condition,Pothole,NaN,NaN,NaN,RISSE STREET,RISSE STREET,PAUL AVENUE,SEDGWICK AVENUE,NaN,NaN,BLOCKFACE,NaN,NaN,NaN,Open,NaN,The Department of Transportation referred this...,2026-03-31T01:58:12.000,Unspecified BRONX,NaN,Unspecified,NaN,BRONX,NaN,NaN,UNKNOWN,Unspecified,BRONX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
424,68508243,2026-03-30T23:51:04.000,2026-03-31T00:40:05.000,NYPD,New York City Police Department,Encampment,NaN,NaN,Subway,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Closed,NaN,The New York City Police Department responded ...,2026-03-31T00:40:10.000,Unspecified MANHATTAN,NaN,Unspecified,NaN,MANHATTAN,987638.0,201032.0,MOBILE,Unspecified,MANHATTAN,NaN,NaN,NaN,F,F Uptown & Queens,NaN,Platform,40.718462,-73.987778,POINT (-73.987777955856 40.718462481372)
721,68510470,2026-03-30T23:04:00.000,NaN,DOT,Department of Transportation,Street Light Condition,Lamppost Knocked Down,Location Type: Intersection,NaN,NaN,NaN,NaN,NaN,NaN,ROCKAWAY BLVD,VAN WYCK EXPY,INTERSECTION,NaN,NaN,NaN,Open,NaN,NaN,NaN,Unspecified QUEENS,NaN,Unspecified,NaN,QUEENS,NaN,NaN,UNKNOWN,Unspecified,QUEENS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
911,68509450,2026-03-30T22:38:07.000,NaN,DOT,Department of Transportation,Street Condition,Pothole,NaN,NaN,NaN,31 STREET,31 STREET,ASTORIA BOULEVARD,ASTORIA BOULEVARD,NaN,NaN,BLOCKFACE,NaN,NaN,NaN,Open,NaN,The Department of Transportation referred this...,2026-03-30T22:38:07.000,Unspecified QUEENS,NaN,Unspecified,NaN,QUEENS,NaN,NaN,UNKNOWN,Unspecified,QUEENS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
with pd.option_context('display.max_rows',None,
                       'display.max_columns',None):
    display(df.head())

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,incident_zip,incident_address,street_name,cross_street_1,cross_street_2,intersection_street_1,intersection_street_2,address_type,city,landmark,facility_type,status,due_date,resolution_description,resolution_action_updated_date,community_board,council_district,police_precinct,bbl,borough,x_coordinate_state_plane,y_coordinate_state_plane,open_data_channel_type,park_facility_name,park_borough,vehicle_type,taxi_company_borough,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,road_ramp,bridge_highway_segment,latitude,longitude,location
0,68509446,2026-03-31T02:39:38.000,NaN,DOT,Department of Transportation,Street Condition,Pothole,NaN,NaN,NaN,65 ROAD,65 ROAD,POWER ROAD,102 STREET,NaN,NaN,BLOCKFACE,NaN,NaN,NaN,Open,NaN,The Department of Transportation referred this...,2026-03-31T02:39:38.000,Unspecified QUEENS,NaN,Unspecified,NaN,QUEENS,NaN,NaN,UNKNOWN,Unspecified,QUEENS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,68506247,2026-03-31T02:30:56.000,NaN,DOT,Department of Transportation,Street Condition,Pothole,NaN,NaN,10031.0,CONVENT AVENUE,CONVENT AVENUE,WEST 133 STREET,WEST 135 STREET,NaN,NaN,BLOCKFACE,MANHATTAN,NaN,NaN,Open,NaN,The Department of Transportation referred this...,2026-03-31T02:30:56.000,09 MANHATTAN,NaN,Precinct 26,NaN,MANHATTAN,NaN,NaN,UNKNOWN,Unspecified,MANHATTAN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,68503135,2026-03-31T02:26:57.000,NaN,DOT,Department of Transportation,Street Condition,Pothole,NaN,NaN,11223.0,NaN,NaN,NaN,NaN,AVENUE S,WEST 7 STREET,INTERSECTION,BROOKLYN,NaN,NaN,Open,NaN,The Department of Transportation referred this...,2026-03-31T02:26:57.000,11 BROOKLYN,43.0,Precinct 62,NaN,BROOKLYN,989974.0,158253.0,UNKNOWN,Unspecified,BROOKLYN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.601042,-73.979387,POINT (-73.979387264647 40.601042365906)
3,68501976,2026-03-31T02:26:00.000,2026-03-31T02:59:00.000,DSNY,Department of Sanitation,Derelict Vehicles,Derelict Vehicles,NaN,Street,11373.0,42-71 79 STREET,79 STREET,WOODSIDE AVENUE,45 AVENUE,NaN,NaN,ADDRESS,ELMHURST,NaN,NaN,Closed,NaN,DSNY already has a request to investigate this...,2026-03-31T12:00:00.000,04 QUEENS,25.0,Precinct 110,4.015250e+09,QUEENS,1015808.0,210118.0,PHONE,Unspecified,QUEENS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.743346,-73.886114,POINT (-73.886113509027 40.74334566695)
4,68509447,2026-03-31T02:17:11.000,NaN,DOT,Department of Transportation,Street Condition,Pothole,NaN,NaN,11101.0,43 STREET,43 STREET,35 AVENUE,NORTHERN BOULEVARD,NaN,NaN,BLOCKFACE,QUEENS,NaN,NaN,Open,NaN,The Department of Transportation referred this...,2026-03-31T02:17:11.000,01 QUEENS,NaN,Precinct 114,NaN,QUEENS,NaN,NaN,UNKNOWN,Unspecified,QUEENS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
subset2

,street_name,cross_street_1,cross_street_2,intersection_street_1,intersection_street_2,address_type,city,landmark,facility_type,status
0,65 ROAD,POWER ROAD,102 STREET,NaN,NaN,BLOCKFACE,NaN,NaN,NaN,Open
1,CONVENT AVENUE,WEST 133 STREET,WEST 135 STREET,NaN,NaN,BLOCKFACE,MANHATTAN,NaN,NaN,Open
2,NaN,NaN,NaN,AVENUE S,WEST 7 STREET,INTERSECTION,BROOKLYN,NaN,NaN,Open
3,79 STREET,WOODSIDE AVENUE,45 AVENUE,NaN,NaN,ADDRESS,ELMHURST,NaN,NaN,Closed
4,43 STREET,35 AVENUE,NORTHERN BOULEVARD,NaN,NaN,BLOCKFACE,QUEENS,NaN,NaN,Open


1. Incident address may be empty because its defined as "House number of incident address provided by submitter" , What if the incident didn't happen in a house, I can't mannually check if each of its value is a correct address or not
2. Street name could be empty cause its defined as  "Street name of incident address provided by the submitter" , can't check if a street name is a valid address or not
3. cross_street and intersection_street are dependent 
4. address type is defined as "Type of incident location information available." So if no address provided then it could be nan .
5. city shouldn't have nan values , Maybe when a location is along the border of the city then it can be nan ?  We can populate this using other columns
6. Landmark can definetely be empty cause not every incident location identitfies as a landmark
7. Not every incident location identifies as a facility 
8. status , we have already dealt with this once 

In [40]:
# Start coding from here onwards next time 
td = df.copy()

In [41]:
df.isnull().sum()

unique_key                             0
created_date                           0
closed_date                        41043
agency                                 0
agency_name                            0
complaint_type                         0
descriptor                          2820
descriptor_2                      125067
location_type                      37166
incident_zip                        2642
incident_address                   12277
street_name                        12281
cross_street_1                     63730
cross_street_2                     63705
intersection_street_1              72611
intersection_street_2              72529
address_type                        1705
city                               10626
landmark                           91597
facility_type                     199346
status                                 0
due_date                          199254
resolution_description             13782
resolution_action_updated_date     12180
community_board 

In [42]:
td = td.drop(['unique_key','created_date','closed_date','agency','agency_name','complaint_type','descriptor','descriptor_2','location_type',
         'status','due_date','resolution_description','resolution_action_updated_date','community_board','council_district','police_precinct',
         'bbl','borough','x_coordinate_state_plane','y_coordinate_state_plane','open_data_channel_type','latitude','longitude','location'],axis=1)

In [43]:
td.loc[na_incident_zip]

,incident_zip,incident_address,street_name,cross_street_1,cross_street_2,intersection_street_1,intersection_street_2,address_type,city,landmark,facility_type,park_facility_name,park_borough,vehicle_type,taxi_company_borough,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,road_ramp,bridge_highway_segment
0,NaN,65 ROAD,65 ROAD,POWER ROAD,102 STREET,NaN,NaN,BLOCKFACE,NaN,NaN,NaN,Unspecified,QUEENS,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,NaN,RISSE STREET,RISSE STREET,PAUL AVENUE,SEDGWICK AVENUE,NaN,NaN,BLOCKFACE,NaN,NaN,NaN,Unspecified,BRONX,NaN,NaN,NaN,NaN,NaN,NaN,NaN
424,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unspecified,MANHATTAN,NaN,NaN,NaN,F,F Uptown & Queens,NaN,Platform
721,NaN,NaN,NaN,NaN,NaN,ROCKAWAY BLVD,VAN WYCK EXPY,INTERSECTION,NaN,NaN,NaN,Unspecified,QUEENS,NaN,NaN,NaN,NaN,NaN,NaN,NaN
911,NaN,31 STREET,31 STREET,ASTORIA BOULEVARD,ASTORIA BOULEVARD,NaN,NaN,BLOCKFACE,NaN,NaN,NaN,Unspecified,QUEENS,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199915,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unspecified,QUEENS,NaN,NaN,NaN,Cross Island Pkwy,North/Bronx Bound,Ramp,Bell Blvd (Exit 32)
199969,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unspecified,QUEENS,NaN,NaN,NaN,Cross Island Pkwy,North/Bronx Bound,Roadway,Bell Blvd (Exit 32) - Throgs Neck Br Bronx New...
199973,NaN,NaN,NaN,NaN,NaN,98 STREET,98 STREET,INTERSECTION,NaN,NaN,DSNY Garage,Unspecified,QUEENS,NaN,NaN,NaN,NaN,NaN,NaN,NaN
199977,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unspecified,QUEENS,NaN,NaN,NaN,BQE/Gowanus Expwy,East/Bronx Bound,Roadway,Roosevelt Ave / Broadway (Exit 37) - Broadway ...


For every column ask ,
1. Can this column have nan values? If yes then when?
2. Check if that column has nan only when its supposed to have nan.

Data cleaning workflow
1. Check nan , are the nan's valid?? What is the status of all the nan's , intentional or accident/error ? har ek column ko izzat se time do
2. Will check if we can fill in the other values using other column values
3. We should have a clear idea of all the columns , how they work , when is what filled , when is it not supposed to be filled 

In [44]:
df.created_date

0         2026-03-31T02:39:38.000
1         2026-03-31T02:30:56.000
2         2026-03-31T02:26:57.000
3         2026-03-31T02:26:00.000
4         2026-03-31T02:17:11.000
                   ...           
199995    2026-03-12T12:42:00.000
199996    2026-03-12T12:41:52.000
199997    2026-03-12T12:41:50.000
199998    2026-03-12T12:41:46.000
199999    2026-03-12T12:41:39.000
Name: created_date, Length: 200000, dtype: str

In [45]:
df.created_date = pd.to_datetime(df.created_date,format="%Y-%m-%dT%H:%M:%S.%f")
df.closed_date = pd.to_datetime(df.closed_date,format="%Y-%m-%dT%H:%M:%S.%f")

df["created_date_only"] = df.created_date.dt.date
df["closed_date_only"] = df.closed_date.dt.date

df["created_time_only"] = df.created_date.dt.time
df["closed_time_only"] = df.closed_date.dt.time

df['response_time'] = df['closed_date'] - df['created_date']
df['response_time']

0                    NaT
1                    NaT
2                    NaT
3        0 days 00:33:00
4                    NaT
               ...      
199995   0 days 03:13:00
199996   0 days 00:48:40
199997               NaT
199998   1 days 01:03:04
199999   0 days 20:14:57
Name: response_time, Length: 200000, dtype: timedelta64[us]

In [46]:
df['response_time'] = df['response_time'].dt.total_seconds().apply(lambda x : x / 3600)

In [47]:
df.response_time.agg(['min','max'])

min   -336.000000
max    438.881944
Name: response_time, dtype: float64

In [48]:
df.loc[df['created_date'] > df['closed_date'],['created_date','closed_date']]
# Gotta remove these columns

df = df.drop(df[df['created_date'] > df['closed_date']].index,axis=0)

In [49]:
df.loc[df.response_time < 0.33,['created_date','closed_date','status']]
# Well we don't need these either, these won't help us predict response time
# Setting the minimum response time to 20 minutes because thats a minimum minutes range I expect for some complaints to be realistically resolved

,created_date,closed_date,status
5,2026-03-31 02:14:08,2026-03-31 02:14:08,Closed
7,2026-03-31 02:05:26,2026-03-31 02:05:26,Closed
8,2026-03-31 02:03:02,2026-03-31 02:03:02,Closed
12,2026-03-31 02:01:14,2026-03-31 02:01:14,Closed
26,2026-03-31 01:49:21,2026-03-31 01:49:21,Closed
...,...,...,...
199955,2026-03-12 12:45:48,2026-03-12 12:45:48,Closed
199959,2026-03-12 12:45:08,2026-03-12 13:03:56,Closed
199974,2026-03-12 12:43:42,2026-03-12 12:51:43,Closed
199984,2026-03-12 12:42:44,2026-03-12 12:42:44,Open


In [50]:
for i in df.complaint_type.unique():
    print(i)

Street Condition
Derelict Vehicles
Illegal Parking
Noise - Street/Sidewalk
Traffic
Noise - Residential
Drug Activity
Noise - Vehicle
Blocked Driveway
Dirty Condition
Homeless Person Assistance
Consumer Complaint
Non-Emergency Police Matter
Noise - Park
Noise - Commercial
Cannabis Retailer
Urinating in Public
LinkNYC
Smoking or Vaping
Lost Property
Indoor Air Quality
Graffiti
Litter Basket Request
Curb Condition
Missed Collection
Encampment
Illegal Dumping
Abandoned Vehicle
Animal-Abuse
Drinking
Rodent
Street Light Condition
Street Sign - Damaged
Food Establishment
Vendor Enforcement
Illegal Tree Damage
Abandoned Bike
Damaged Tree
Panhandling
Illegal Posting
Obstruction
GENERAL
For Hire Vehicle Complaint
Noise
UNSANITARY CONDITION
PLUMBING
SAFETY
HEAT/HOT WATER
Outdoor Dining
Traffic Signal Condition
Dead Animal
Sewer
ELECTRIC
Water System
Noise - Helicopter
Food Poisoning
WATER LEAK
Lead
Illegal Fireworks
FLOORING/STAIRS
Hazardous Materials
Overgrown Tree/Branches
Sidewalk Condition
DO

In [51]:
df = df.drop(df[df['response_time'] < 0.33].index,axis=0)

In [52]:
# impute Nat with something so that it means response not taken

# agency, complaint_type , police precinict , address_type 
# lattiude , longitude ( ye mene iss liye pick kiya cause incomplete info can also be the reason why response was slow or not made and these columns use geovalidation , these are empty means that not enough address was provided )



In [53]:
df.agency.unique()

<StringArray>
[  'DOT',  'DSNY',  'NYPD',   'DHS',  'DCWP',   'OOS',   'OTI', 'DOHMH',
   'TLC',   'DPR',   'HPD',   'DEP',   'EDC',   'DOB',   'DOE']
Length: 15, dtype: str

In [54]:
df[['agency','complaint_type','police_precinct','address_type','latitude','longitude']].info()
# great , three of these don't have any null values

<class 'pandas.DataFrame'>
Index: 178062 entries, 0 to 199999
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   agency           178062 non-null  str    
 1   complaint_type   178062 non-null  str    
 2   police_precinct  178062 non-null  str    
 3   address_type     176459 non-null  str    
 4   latitude         170572 non-null  float64
 5   longitude        170572 non-null  float64
dtypes: float64(2), str(4)
memory usage: 9.5 MB


In [55]:
# Since this is small , We can remove these
df = df.drop(df.loc[df.address_type.isna(),'closed_date'].index,axis=0)

In [56]:
df = df.drop(df[df.latitude.isna()].index,axis=0)

In [57]:
# lets first look at the columns which have no non null values
df.loc[:,~df.columns.isin(['agency','complaint_type','police_precinct','address_type','latitude','longitude','unique_key','created_date','closed_date','agency_name'])].info()

<class 'pandas.DataFrame'>
Index: 170099 entries, 2 to 199999
Data columns (total 39 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   descriptor                      167938 non-null  str    
 1   descriptor_2                    71819 non-null   str    
 2   location_type                   149730 non-null  str    
 3   incident_zip                    170068 non-null  float64
 4   incident_address                162553 non-null  str    
 5   street_name                     162550 non-null  str    
 6   cross_street_1                  115934 non-null  str    
 7   cross_street_2                  115986 non-null  str    
 8   intersection_street_1           113147 non-null  str    
 9   intersection_street_2           113224 non-null  str    
 10  city                            163167 non-null  str    
 11  landmark                        98357 non-null   str    
 12  facility_type                   

In [58]:
df.community_board.unique()

<StringArray>
[              '11 BROOKLYN',                 '04 QUEENS',
               '05 BROOKLYN',                 '08 QUEENS',
                 '12 QUEENS',               '13 BROOKLYN',
                  '05 BRONX',                  '01 BRONX',
               '01 BROOKLYN',               '15 BROOKLYN',
                 '01 QUEENS',               '18 BROOKLYN',
              '12 MANHATTAN',               '10 BROOKLYN',
                  '09 BRONX',              '10 MANHATTAN',
               '03 BROOKLYN',                 '05 QUEENS',
          '01 STATEN ISLAND',              '03 MANHATTAN',
                  '06 BRONX',                 '07 QUEENS',
                  '12 BRONX',              '04 MANHATTAN',
                  '03 BRONX',               '02 BROOKLYN',
                  '07 BRONX',               '12 BROOKLYN',
              '11 MANHATTAN',               '04 BROOKLYN',
                  '04 BRONX',                  '08 BRONX',
                 '80 QUEENS',             

In [59]:
df.loc[df['community_board'].str.contains('Unspecified',case=False, regex=True),'community_board'] = 'Unspecified'

In [60]:
freq = df['community_board'].value_counts()
df['community_board_encoded'] = df['community_board'].map(freq)
# encoded each community_board with its frequency , for cleaning yk what I did

In [61]:
df.borough.unique()
# One hot encoding will handle this

<StringArray>
['BROOKLYN', 'QUEENS', 'BRONX', 'MANHATTAN', 'STATEN ISLAND', 'Unspecified']
Length: 6, dtype: str

In [62]:
#df.x_coordinate_state_plane.isna().sum()
# x and y coordinate_state_plane are both clean 

In [63]:
print(df.open_data_channel_type.unique()) 
print(df.park_borough.unique())
# All good encoding can handle these

<StringArray>
['UNKNOWN', 'PHONE', 'MOBILE', 'ONLINE']
Length: 4, dtype: str
<StringArray>
['BROOKLYN', 'QUEENS', 'BRONX', 'MANHATTAN', 'STATEN ISLAND', 'Unspecified']
Length: 6, dtype: str


In [64]:
display(df.resolution_action_updated_date.unique())
# I can make a column like how many days has it been since resolution action update , I think that could help us predict a lot of things

df['resolution_action_updated_date'] = df.resolution_action_updated_date.astype('datetime64[ns]')

<StringArray>
['2026-03-31T02:26:57.000', '2026-03-31T12:00:00.000',
                       nan, '2026-03-31T01:49:43.000',
 '2026-03-31T02:01:42.000', '2026-03-31T01:49:07.000',
 '2026-03-31T01:54:09.000', '2026-03-31T01:25:57.000',
 '2026-03-31T02:01:12.000', '2026-03-31T01:21:26.000',
 ...
 '2026-03-18T13:09:25.000', '2026-03-12T14:17:05.000',
 '2026-03-12T14:36:59.000', '2026-03-12T13:12:58.000',
 '2026-03-12T22:56:31.000', '2026-03-12T13:57:09.000',
 '2026-03-12T13:52:44.000', '2026-03-12T15:55:00.000',
 '2026-03-12T13:30:38.000', '2026-03-13T13:44:55.000']
Length: 96760, dtype: str

In [65]:
df.resolution_action_updated_date

2        2026-03-31 02:26:57
3        2026-03-31 12:00:00
6                        NaT
9                        NaT
10                       NaT
                 ...        
199995   2026-03-12 15:55:00
199996   2026-03-12 13:30:38
199997   2026-03-12 00:00:00
199998   2026-03-13 13:44:55
199999   2026-03-13 00:00:00
Name: resolution_action_updated_date, Length: 170099, dtype: datetime64[ns]

In [66]:
df['time_since_last_resolution_action_day'] = df.resolution_action_updated_date.dt.day
df['time_since_last_resolution_action_hour'] = (df.resolution_action_updated_date - df.created_date).dt.total_seconds() // 3600

In [67]:
df.time_since_last_resolution_action_hour.min()
# Wtf is this value 

np.float64(-31568.0)

In [68]:
df.loc[:,['created_date','resolution_action_updated_date','time_since_last_resolution_action_hour']].min()
# This is either a glitch or it has something to do with the inner workings 
# But there is no way for us to know what those inner workings are so , we will have to remove these -ve rows

created_date                              2026-03-12 12:41:39
resolution_action_updated_date            2022-08-17 08:58:46
time_since_last_resolution_action_hour               -31568.0
dtype: object

In [69]:
df.loc[df.time_since_last_resolution_action_hour <= 0].shape # That is a lot of columns we can't just drop these
# We will need to impute these with something

(49714, 52)

We need to impute these with a value such that linear regression won't try to learn from them 

In [70]:
closed_date_x_status
            

,closed_date,status,Possible?,description
0,not_null,Open,False,Human error
1,not_null,Closed,True,Possible
2,not_null,NaN,True,Human error
3,not_null,Assigned,False,Human error
4,not_null,Started,False,Human error
5,not_null,Pending,False,Human error
6,not_null,Unspecified,True,Human error
7,NaN,Open,True,Possible
8,NaN,NaN,False,Human error
9,NaN,In Progress,True,Possible


In [71]:
closed_date_x_status.iloc[6,1]

'Unspecified'

In [72]:
df.iloc[(df['closed_date'].notna()) & (df['status']==closed_date_x_status.iloc[1,1])]

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,incident_zip,...,longitude,location,created_date_only,closed_date_only,created_time_only,closed_time_only,response_time,community_board_encoded,time_since_last_resolution_action_day,time_since_last_resolution_action_hour
3,68501976,2026-03-31 02:26:00,2026-03-31 02:59:00,DSNY,Department of Sanitation,Derelict Vehicles,Derelict Vehicles,NaN,Street,11373.0,...,-73.886114,POINT (-73.886113509027 40.74334566695),2026-03-31,2026-03-31,02:26:00,02:59:00,0.550000,2497,31.0,9.0
35,68501219,2026-03-31 01:41:31,2026-03-31 02:01:44,NYPD,New York City Police Department,Noise - Park,Loud Talking,NaN,Park/Playground,11223.0,...,-73.970650,POINT (-73.970649628629 40.587730936142),2026-03-31,2026-03-31,01:41:31,02:01:44,0.336944,1454,31.0,0.0
66,68501278,2026-03-31 01:28:48,2026-03-31 01:49:02,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,11232.0,...,-73.996418,POINT (-73.99641815822 40.644691824248),2026-03-31,2026-03-31,01:28:48,01:49:02,0.337222,2919,31.0,0.0
69,68504275,2026-03-31 01:25:58,2026-03-31 01:54:14,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,10037.0,...,-73.936127,POINT (-73.936127084377 40.811923091319),2026-03-31,2026-03-31,01:25:58,01:54:14,0.471111,2368,31.0,0.0
75,68504007,2026-03-31 01:23:55,2026-03-31 02:01:26,NYPD,New York City Police Department,Noise - Commercial,Loud Talking,NaN,Club/Bar/Restaurant,11216.0,...,-73.941338,POINT (-73.94133816146 40.682793665961),2026-03-31,2026-03-31,01:23:55,02:01:26,0.625278,3241,31.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199994,68300565,2026-03-12 12:42:09,2026-03-12 13:52:38,NYPD,New York City Police Department,Encampment,NaN,NaN,Street/Sidewalk,10007.0,...,-74.002731,POINT (-74.002730603521 40.712679886843),2026-03-12,2026-03-12,12:42:09,13:52:38,1.174722,1362,12.0,1.0
199995,68303736,2026-03-12 12:42:00,2026-03-12 15:55:00,DEP,Department of Environmental Protection,Water System,Possible Water Main Break (Use Comments) (WA1),WA1,NaN,11203.0,...,-73.926201,POINT (-73.926201498738 40.654085555145),2026-03-12,2026-03-12,12:42:00,15:55:00,3.216667,3210,12.0,3.0
199996,68298047,2026-03-12 12:41:52,2026-03-12 13:30:32,NYPD,New York City Police Department,Illegal Parking,Posted Parking Sign Violation,NaN,Street/Sidewalk,11249.0,...,-73.960261,POINT (-73.960261410034 40.716631006337),2026-03-12,2026-03-12,12:41:52,13:30:32,0.811111,4246,12.0,0.0
199998,68303589,2026-03-12 12:41:46,2026-03-13 13:44:50,DSNY,Department of Sanitation,Dirty Condition,Trash,Not Cleaned by Property Owner,Yard,11228.0,...,-74.010250,POINT (-74.010250409554 40.611852762534),2026-03-12,2026-03-13,12:41:46,13:44:50,25.051111,3496,13.0,25.0


In [73]:
L = []
td = closed_date_x_status
for i in range(14):
    if td.iloc[i,3]=='Human error':
        if td.iloc[i,0]=='not_null':
            if td.iloc[i,1]!=np.nan:
                L.append(df.iloc[(df['closed_date'].notna()) & (df['status']==closed_date_x_status.iloc[i,1])].shape[0])
            else:
                L.append(0)
        else:
            if td.iloc[i,1]!=np.nan:
                L.append(df.iloc[(df['closed_date'].isna()) & (df['status']==closed_date_x_status.iloc[1,1])].shape[0])
            else:
                L.append(0)
    else:
        L.append(np.nan)

closed_date_x_status['count'] = L
print('Size of not possible values:',closed_date_x_status['count'].sum())
print("Size of actual possible values :",df.shape[0] - closed_date_x_status['count'].sum())
    

Size of not possible values: 498.0
Size of actual possible values : 169601.0


In [74]:
closed_date_x_status

,closed_date,status,Possible?,description,count
0,not_null,Open,False,Human error,2.0
1,not_null,Closed,True,Possible,NaN
2,not_null,NaN,True,Human error,0.0
3,not_null,Assigned,False,Human error,496.0
4,not_null,Started,False,Human error,0.0
5,not_null,Pending,False,Human error,0.0
6,not_null,Unspecified,True,Human error,0.0
7,NaN,Open,True,Possible,NaN
8,NaN,NaN,False,Human error,0.0
9,NaN,In Progress,True,Possible,NaN


In [75]:
df['status'].isin(['Assigned','Open'])

2          True
3         False
6         False
9         False
10        False
          ...  
199995    False
199996    False
199997     True
199998    False
199999    False
Name: status, Length: 170099, dtype: bool

In [76]:
df[df['closed_date'].notnull()].index

Index([     3,     35,     66,     69,     75,    101,    104,    105,    109,
          110,
       ...
       199986, 199987, 199988, 199991, 199992, 199994, 199995, 199996, 199998,
       199999],
      dtype='int64', length=132091)

In [77]:
# This is very small , Removing this won't affect our data much
# Had to check/confirm how to do this in a single like 
# Encountered the memory error for the first time , had to consult ai if my logic was correct or not
df = df.drop(df[(df['closed_date'].notnull()) & (df['status'].isin(['Assigned','Open']))].index,axis=0)


In [78]:
df.council_district.unique()

array([43., 25., 37., 24., 27., 47., 14., 17., 33., 44., 22., 46., 21.,
       10., 18.,  9., 41., 28., 34., 30., 49.,  1., 15., 20., 12.,  3.,
       16., 23., 38., 36., 35., 19.,  8., 32., 29.,  2., 48., 11.,  6.,
       39., 40., 50., 13., 31., 42., 45.,  7., 26.,  4., 51., nan,  5.])

In [79]:
df.council_district.isna().sum()
# Small again , we can remove this
df = df.drop(df[df.council_district.isna()].index,axis=0)

In [80]:
df.bbl.agg(['min','max'])

min    0.000000e+00
max    5.270001e+09
Name: bbl, dtype: float64

In [81]:
df.bbl.value_counts()

bbl
2.025110e+09    315
4.142600e+09    275
3.063150e+09    261
2.046990e+09    255
4.068290e+09    227
               ... 
3.013310e+09      1
3.068480e+09      1
4.119470e+09      1
1.001590e+09      1
3.046600e+09      1
Name: count, Length: 62017, dtype: int64

In [82]:
df.bbl.isna().sum()
# Can't remove these 
# what do we do? frequency encoding?
# I am thinking of imputing NaN with lets say 0 

np.int64(18402)

In [83]:
df.taxi_company_borough.unique()
df.bridge_highway_name.unique()
# probably onehot encoding , or freq encoding or i just Impute with something , Idk what to impute with

<StringArray>
[                      nan,       'Long Island Expwy',
          'Van Wyck Expwy', 'Henry Hudson Pkwy/Rt 9A',
                  'FDR Dr',   'Queensboro/59th St Br',
      'Grand Central Pkwy',               'Belt Pkwy',
      'Major Deegan Expwy',       'BQE/Gowanus Expwy',
         'Harlem River Dr',          'FDR Southbound']
Length: 12, dtype: str

In [84]:
df.due_date.isna().sum()

np.int64(168347)

In [ ]:
df.facility_type.unique()
# Since there are only two values , i think i can encode it with 1 and 0 , represention not null and null value

<StringArray>
[nan, 'DSNY Garage']
Length: 2, dtype: str

In [ ]:
df.landmark.unique()
# I think frequency encoding should do it

<StringArray>
[                   nan,     'ARLINGTON AVENUE',             '87 DRIVE',
           '149 STREET',      'BAY VIEW AVENUE',        'WALTON AVENUE',
      'EAST 152 STREET',          'WEST STREET',             'AVENUE U',
              '25 ROAD',
 ...
          'FRIEL PLACE',          'HAVEN PLAZA', 'ATLANTIC CENTER MALL',
       'LUCILLE AVENUE',          'PAUW STREET',          'ZEBRA PLACE',
         'MOSCO STREET',      'BEACH 64 STREET',          'JULIUS ROAD',
     'ST ANDREWS PLAZA']
Length: 5206, dtype: str

In [ ]:
df.city.unique()
# I have no idea , frequency encoding??
# Target encoding? but that supposed to be done on the train dataset only
# Nan becomes its own category

<StringArray>
[           'BROOKLYN',            'ELMHURST',             'JAMAICA',
               'BRONX',                   nan,             'ASTORIA',
              'CORONA',            'NEW YORK',           'RIDGEWOOD',
       'STATEN ISLAND',            'FLUSHING',       'FRESH MEADOWS',
       'EAST ELMHURST',              'QUEENS',             'BAYSIDE',
          'OZONE PARK',       'RICHMOND HILL',        'HOWARD BEACH',
       'COLLEGE POINT',      'QUEENS VILLAGE',        'FOREST HILLS',
             'MASPETH',    'SOUTH OZONE PARK',     'JACKSON HEIGHTS',
            'WOODSIDE',       'NEW HYDE PARK',     'OAKLAND GARDENS',
           'SUNNYSIDE',           'REGO PARK',         'KEW GARDENS',
           'MANHATTAN',      'MIDDLE VILLAGE',             'ARVERNE',
 'SOUTH RICHMOND HILL',            'ROSEDALE',          'WHITESTONE',
           'WOODHAVEN',    'LONG ISLAND CITY',        'FAR ROCKAWAY',
     'CAMBRIA HEIGHTS',           'BELLEROSE', 'SPRINGFIELD GARDENS',
      

In [ ]:
df.taxi_company_borough.unique()

<StringArray>
[nan, 'BROOKLYN', 'QUEENS', 'BRONX', 'MANHATTAN', 'STATEN ISLAND']
Length: 6, dtype: str

In [111]:
df.borough.notna().shape

(169091,)

In [ ]:
df.location_type.value_counts()['Taxi']
# taxi_company_borough or taxi_pickup_location may be empty but location_type = Taxi cannot be false

np.int64(330)

In [124]:
df.taxi_company_borough.notnull().sum()

np.int64(143)

In [129]:
df.vehicle_type.notna().sum()

np.int64(8269)

In [ ]:
print(df.loc[(df['borough'].notnull()) & (df['taxi_company_borough'].notnull())].shape)
print(df.loc[(df['borough'].isnull()) & (df['taxi_company_borough'].notnull())].shape)
print(df.loc[(df['borough'].isnull()) & (df['taxi_company_borough'].isnull())].shape)
print(df.loc[(df['borough'].notnull()) & (df['taxi_company_borough'].isnull())].shape)
# Just checking this cause both are borough , so i was wondering if both are given at a time

(143, 52)
(0, 52)
(0, 52)
(168948, 52)


- We have 330 taxi records and out of which , only 143 taxi borough records . 
- The amount of data is insufficient to predict even location type taxi , much less extract patterns from taxi company borough 
- Facility , landmark and highway | road ramp are in similar condtions
- Therefore, I believe it would be sufficient to have a column with category taxi , highway, landmark , facility , other

In [130]:
df.vehicle_type.unique()

<StringArray>
[                      nan,                     'Car',
                     'SUV',                     'Van',
                   'Truck',                   'Other',
             'Car Service', 'Ambulette / Paratransit',
            'Commuter Van']
Length: 9, dtype: str

In [ ]:
# For vehicle_type we can idk one hot encode it i guess , There are way too many nulls can't remove them
# Target encoding when using Random forest

In [85]:
# community_board , borough , x_coordinate_state_plane ,y_coordinate_state_plane , open_data_channel_type , park_borough done

In [86]:
# resolution_action_updated_date , status ,  council_district , bbl , vechicle_type , taxi_company_borough , bride_highway_name 

In [88]:
#  landmark , facility_type , city , due_date

In [89]:
# Now I pick 6 more Columns other than the 6 columns I already picked and think about what to do with their NaN
# Pick 6 columns other than the one you already picked that can help you predict response time


In [91]:
#workflow of how a record is added and when is it confirmed

#### Hide

##### 4. EDA

$$\huge \texttt{Exploratory data analysis}$$

In [92]:
# Columns that I think might affect Response time
claim = [{'refers to the same agency':['agency','agency']},
         'complaint_type',
         'due_date',
         'resolution_description',
         'resolution_action_updated_date',
         'open_data_channel_type',
         {'description of problem':['descriptor','descriptor_2']},
         {'location':   ['location_type','incident_zip','incident_address', 'street_name', 'cross_street_1', 'cross_street_2',
                            'intersection_street_1', 'intersection_street_2','address_type','city', 'landmark', 'facility_type',
                            'community_board', 'council_district','police_precinct', 'bbl', 'borough', 'x_coordinate_state_plane',
                            'y_coordinate_state_plane','latitude', 'longitude', 'location'],
       'park_location': ['park_facility_name', 'park_borough'],
       'taxi_location': ['taxi_company_borough', 'taxi_pick_up_location'],
       'bridge_location': ['bridge_highway_name',
                            'bridge_highway_direction', 'road_ramp', 'bridge_highway_segment']},]

# Columns that I don't think affect the Response time
NotClaim = ['unique_key', 'created_date', 'closed_date','status']

##### 5. Data transformations

$$\huge \texttt{Data pre-processing}$$
$$\normalsize \texttt{In this section , We are gonna prepare our data for model input}$$

In [93]:
# OneHot-Encoding all the categorical columns at once
"""
X_train , X_test , y_train , y_test = train_test_split(...)

X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)

# Train and test might have different categories 
#ex Train has : A, B and Test has : A, C , so columns don't match
#This makes sure , both have same columns , missing ones are filled with 0
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)
"""

"\nX_train , X_test , y_train , y_test = train_test_split(...)\n\nX_train = pd.get_dummies(X_train)\nX_test = pd.get_dummies(X_test)\n\n# Train and test might have different categories \n#ex Train has : A, B and Test has : A, C , so columns don't match\n#This makes sure , both have same columns , missing ones are filled with 0\nX_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)\n"

##### 6. Training

$$\huge \texttt{Model Training}$$
$$\normalsize \texttt{Training the model / trying out different things with the model and improving the result of desired metrics}$$

##### 7. evaluation

$$\huge \texttt{Model evaluation / interpretation}$$
$$\normalsize \texttt{We showcase and interpret the result here}$$